# Structural missingness audit

This supporting audit gives the training and test predictor frames one consistent first-pass missingness check. It distinguishes pandas nulls from blank source strings and deliberately leaves feature-specific semantic sentinels, such as zero coordinates or `unknown`, to the focused predictor audits.

## 1. Load and validate the immutable source frames

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


stage_directory = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data" / "TrainingSetValues.csv").is_file()
)
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import column_missingness_summary, frame_missingness_summary
from source_data_validation import validate_raw_feature_schema

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
frames = {"training": training_features, "test": test_features}

print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows."
)

Validated 59,400 training rows and 14,850 test rows.


## 2. Measure missing cells and affected rows

The mean missing cells per affected row is the interpretable version of the ratio used in the Module 7 exercise. The surrounding counts and percentages make clear what that ratio does and does not describe.

In [2]:
frame_summary = frame_missingness_summary(frames)
display(frame_summary)

,rows,columns,cells,explicit missing cells,source blank cells,structural missing cells,structural missing cells (%),rows with structural missing,rows with structural missing (%),mean missing cells per affected row,maximum missing cells in one row,complete rows
frame,,,,,,,,,,,,
training,59400,40,2376000,0,46094,46094,1.94,31587,53.18,1.459,6,27813
test,14850,40,594000,0,11464,11464,1.93,7906,53.24,1.450,6,6944


## 3. Locate structural missingness by feature

The raw CSVs are loaded with `keep_default_na=False`, so source blanks remain distinguishable from pandas nulls. Only features with at least one structurally missing value are shown below.

In [3]:
column_summary = column_missingness_summary(frames)
structurally_missing = column_summary.loc[
    column_summary["structural missing"].gt(0)
]
display(structurally_missing)

,,dtype,rows,explicit missing,explicit missing (%),source blank,source blank (%),structural missing,structural missing (%),non-missing unique
feature,frame,,,,,,,,,
funder,training,str,59400,0,0.0,3635,6.12,3635,6.12,1897
installer,training,str,59400,0,0.0,3655,6.15,3655,6.15,2145
subvillage,training,str,59400,0,0.0,371,0.62,371,0.62,19287
public_meeting,training,str,59400,0,0.0,3334,5.61,3334,5.61,2
scheme_management,training,str,59400,0,0.0,3877,6.53,3877,6.53,12
scheme_name,training,str,59400,0,0.0,28166,47.42,28166,47.42,2696
permit,training,str,59400,0,0.0,3056,5.14,3056,5.14,2
funder,test,str,14850,0,0.0,869,5.85,869,5.85,980
installer,test,str,14850,0,0.0,877,5.91,877,5.91,1091


## 4. Compare training and test missingness

Large percentage-point differences would suggest a collection or coverage shift that the later preprocessing pipeline must handle.

In [4]:
missingness_comparison = (
    column_summary["structural missing (%)"]
    .unstack("frame")
    .loc[:, ["training", "test"]]
)
missingness_comparison["test minus training (pp)"] = (
    missingness_comparison["test"] - missingness_comparison["training"]
)
missingness_comparison = missingness_comparison.loc[
    missingness_comparison[["training", "test"]].max(axis=1).gt(0)
].sort_values("training", ascending=False)
display(missingness_comparison)

frame,training,test,test minus training (pp)
feature,,,
scheme_name,47.42,47.76,0.34
scheme_management,6.53,6.53,0.00
installer,6.15,5.91,-0.24
funder,6.12,5.85,-0.27
public_meeting,5.61,5.53,-0.08
permit,5.14,4.96,-0.18
subvillage,0.62,0.67,0.05


## Conclusions

- Structural missing cells occupy about 1.9% of each predictor frame, but they affect just over 53% of rows because blanks occur across several features.
- An affected row contains about 1.45 structurally missing cells on average, with at most six in either supplied frame.
- `scheme_name` dominates the structural count; the remaining affected features have much lower blank rates.
- Training and test rates are closely aligned. The largest feature-level difference is well below one percentage point.
- This structural view is not the complete missing-data model. Numeric zeros, literal sentinel tokens and invalid combinations remain visible for the feature-specific audits to interpret.